# Page 6 — "Thinking Longer": the length of the chain is the compute

Cap the model's reasoning at N tokens, force it to commit to an answer at the cap, grade what it
committed to, sweep N. Greedy throughout, so the budget is the only variable.

**This continues page 5 on the same instrument.** Page 5 measured both ends of the curve drawn
here — forbidden to reason (7% on MATH L4-5) and unconstrained (43%) — on these same models, items
and grader. Page 6 fills in the middle.

Writes `out/page_06/`: `traces.jsonl`, `derived.json`, `manifest.json`.

Budget grid, model choice and the claims contract live in
[`docs/page_6_refactor.md`](../docs/page_6_refactor.md). Shared plumbing lives in
[`harness.py`](harness.py), which ships alongside this notebook.

## 0. How to run this

| Mode | Models | Items | Budgets | Hardware | Time |
|---|---|---|---|---|---|
| `QUICK` | Qwen2.5-3B | 30 | 128 · 512 · 2048 | ~6 GB GPU | ~20 min |
| `FULL` | 1.5B + 3B + 7B | 100 | 128 · 256 · 512 · 1024 · 2048 | 16 GB+ GPU | 2-3 h |

**The grid is derived, not guessed.** From the same models' unconstrained traces: at 128 tokens
almost nothing has finished, at 512 about 45% has, and at 2048 between 94% and 99% has. Five
budgets cover the transition with nothing spent below it. The top is marginally binding — section
7's pilot reports how much is still running there and says when to add 4096.

**Three curves, one tokenizer.** All three checkpoints hash to the same tokenizer, which is what
lets them share a token-budget x-axis. Qwen3 does not, and cannot be a fourth line on this chart
whatever else it has to recommend it.

Resumable: generations are appended to `out/page_06/traces.jsonl` and keyed by `RUN_ID`. A problem
that finished inside a smaller cap is carried forward rather than regenerated, which is most of why
`FULL` is hours instead of days.

```bash
pip install "torch>=2.7" "transformers>=4.56" accelerate bitsandbytes sympy matplotlib numpy requests
pip install reasoning-from-scratch
```

# Part 1 — Setup and the sweep

Configuration, the item set, the prompts, the model, the generations and the checks on them.
Everything here produces `traces.jsonl` and nothing here draws anything.

**Skip to part 2** if what you came for is how the page's figures and numbers were made. In
`REPLAY` this part loads no model and takes about a minute; you are not skipping a wait.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "FULL"       # "REPLAY" (no GPU), "QUICK" (one 3B), "FULL" (three curves)
SEED = 123                # selects items only; decoding is greedy and unseeded
DATASET = "math500_l45"   # or "gsm8k" -- both are drawn below, this picks the swept one

# Page 5's models, so the two pages are one measurement. The 7B is 8-bit because
# 15 GB of bf16 does not fit 16 GB; quantisation is constant across budgets, so it
# cannot bend a curve, only shift its level -- and the level is not the claim.
PRIMARY = "Qwen/Qwen2.5-3B-Instruct"
CURVES_FULL = [(PRIMARY, "bf16"),
               ("Qwen/Qwen2.5-1.5B-Instruct", "bf16"),
               ("Qwen/Qwen2.5-7B-Instruct", "8bit")]

PRESETS = {
    # Every figure from the published generations: no GPU, no weights, about a minute.
    "REPLAY": dict(models=CURVES_FULL, n_items=100, budgets=[128, 256, 512, 1024, 2048],
                   waits=[0, 1, 2, 3]),
    "QUICK":  dict(models=[(PRIMARY, "bf16")], n_items=30, budgets=[128, 512, 2048],
                   waits=[0, 1, 2]),
    "FULL":   dict(models=CURVES_FULL, n_items=100, budgets=[128, 256, 512, 1024, 2048],
                   waits=[0, 1, 2, 3]),
}
CFG = PRESETS[RUN_MODE]
MODELS = CFG["models"]

# Log-spaced, and bounded by measurement rather than by taste: on these checkpoints
# 0% of unconstrained traces finish inside 128 tokens and 100% finish inside 2048, so
# a budget outside that range is either all-forced or non-binding.
BUDGETS = sorted(CFG["budgets"])
MAX_BUDGET = BUDGETS[-1]

# Tokens allowed after "\boxed{" is handed over: enough to write an answer and shut
# the brace, not enough to resume reasoning. Section 10 check 2 reports how often it
# was too few.
ANSWER_ALLOWANCE = 32

# The other arm (section 11). Budget forcing has two directions: cut the thinking
# short, or refuse to let it stop. Everything above is the first. WAITS counts how
# many times we suppress the end of the turn and append the word; k=0 is the model
# left alone, and its records must match the uncapped sweep above.
WAIT_WORD = "Wait"
WAITS = sorted(CFG["waits"])
WAIT_ALLOWANCE = 512       # tokens each injection buys; k=0 gets MAX_BUDGET instead,
                           # because k=0 is the model left alone and must match the
                           # top of the sweep above rather than being capped again

# Where REPLAY reads its generations. A local out/page_06/traces.jsonl wins if present.
TRACES_URL = None          # set to the published traces.jsonl once it is hosted

if RUN_MODE == "FULL":
    print("!" * 76)
    print("WARNING  A full reproduction sweeps three checkpoints over five budgets, then")
    print("         runs the wait arm over the same items. Hours on a 16 GB+ GPU.")
    print("         RUN_MODE = \"REPLAY\" draws every")
    print("         figure on this page from the published traces in about a minute.")
    print("!" * 76 + "\n")

print(f"mode={RUN_MODE}  dataset={DATASET}  items={CFG['n_items']}")
print(f"budgets={BUDGETS}   answer allowance={ANSWER_ALLOWANCE}")
print(f'waits={WAITS}   injecting "{WAIT_WORD}"   allowance {WAIT_ALLOWANCE}/injection')
for mid, q in MODELS:
    print(f"   {mid.split('/')[-1]:<24}{q}")

mode=REPLAY  dataset=math500_l45  items=100
budgets=[128, 256, 512, 1024, 2048]   answer allowance=32
   Qwen2.5-3B-Instruct     bf16
   Qwen2.5-1.5B-Instruct   bf16
   Qwen2.5-7B-Instruct     8bit


In [2]:
import json
import random
import re
import textwrap
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm.auto import tqdm

# Shared plumbing, shipped beside this notebook.
import harness

# The inherited grading harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_06")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = harness.mean
median = harness.median

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

/home/zenithblade/workspace/research/sapiotomy/notebooks/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0+cu128
harness 1.2 ab45365b63ab


## 1. The grading harness

Inherited unchanged from ch03. `extract_final_candidate` takes the last `\boxed{...}` (falling
back to the last number); `grade_answer` checks equivalence with LaTeX canonicalisation and
SymPy. The next cell adds the two shapes this page generates: a trace cut mid-sentence then
forced to commit, and one cut with no forcing, which must grade wrong because there is nothing
to extract.

In [3]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                             "9",               True),
    (r"27",                                                    "9",               False),  # wrong blurt
    (r"\boxed{\frac{1}{2}}",                                  "0.5",             True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                        "1024",            True),   # thousands separator
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}",              r"\left( 3, \frac{\pi}{2} \right)", True),

    # --- the two shapes page 6 generates -----------------------------------
    # cut mid-sentence, think block closed for it, then forced to commit:
    ("Common denominator is 12, so 3/4 = 9/12 and 2/3 = 8/12. Adding gives 17/12. "
     "Now I need to subtr\n</think>\n\nFinal answer: " + r"\boxed{\frac{17}{12}}",
     r"\frac{17}{12}",  True),
    # cut mid-sentence with no forcing: nothing to extract, so it must grade wrong.
    ("Common denominator is 12, so 3/4 = 9/12 and 2/3 = 8/12. Now I need to subtr",
     r"\frac{17}{12}",  False),
    # forced, but the model committed to the wrong thing -- forcing must not rescue it.
    ("...\n</think>\n\nFinal answer: " + r"\boxed{\frac{5}{12}}", r"\frac{17}{12}", False),
]

rows = [(raw, gt, expected, bool(grade_answer(extract_final_candidate(raw), gt)))
        for raw, gt, expected in CHECKS]
failures = [r for r in rows if r[3] != r[2]]

if failures:                                   # the table is for debugging, not for reading
    for raw, gt, expected, got in rows:
        flag = "ok " if got == expected else "!! "
        shown = raw if len(raw) < 58 else raw[:30] + " … " + raw[-25:]
        print(f"{flag}{shown:<60} -> {extract_final_candidate(raw)!r:<22} vs {gt!r:<20} {got}")
    raise AssertionError(f"{len(failures)} grader check(s) behaved unexpectedly")

print(f"grader preflight: {len(rows)}/{len(rows)} as expected, including the two shapes this page "
      f"generates (cut then forced; cut with nothing to extract)")


grader preflight: 8/8 as expected, including the two shapes this page generates (cut then forced; cut with nothing to extract)


## 2. The item set

MATH-500, levels 4 and 5. Drawn with a fixed seed and hashed, so "the same items" is checkable
later rather than asserted. The seed selects items and nothing else: decoding is greedy.

The figures only exist if the model fails at low budget and succeeds at high budget. Section 7
measures that on this model rather than assuming it.

In [4]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

BANDS = {
    # MATH levels 4-5: page 5 measured this band's endpoints at 7% and 43%.
    "math500_l45": [r for r in math500 if int(r["level"]) in (4, 5)],
    # GSM8K: a wider climb (15% to 71%) over a narrower budget range, because its
    # traces are a third as long. Both are drawn; DATASET picks the one swept.
    "gsm8k": harness.load_gsm8k(DATA_DIR / "gsm8k_test.jsonl"),
}

band = sorted(BANDS[DATASET], key=lambda r: r["unique_id"])
random.Random(SEED).shuffle(band)
items = band[: CFG["n_items"]]
assert len(items) == CFG["n_items"], \
    f"asked for {CFG['n_items']} items, {DATASET} holds only {len(band)}"

ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)

from collections import Counter
spread = "+".join(f"L{n}x{c}" for n, c in sorted(Counter(int(r["level"]) for r in items).items()))
print(f"{DATASET}: {len(items)} of {len(band)} available   {spread}   "
      f"sha256 {ITEM_SET_HASH[:12]}")

math500_l45: 100 of 262 available   L4x50+L5x50   sha256 9ab2b5792042


## 3. The models

Qwen3, because the intervention needs a think block that can be cut.

**Both curves come from the same family, so they share a tokenizer.** Token counts are this
page's x-axis, which is why section 4 derives the `</think>` and end-of-turn ids once and reuses
them for both models. It is the first thing that breaks if the ladder leaves the Qwen3 family.

bf16 on GPU, fp32 on CPU; the harness checks the dtype the weights actually loaded with.

## Replaying the published run

**`REPLAY` reads generations instead of making them.** Nothing below knows the difference: the
same `results` structure is built either way, so every figure, check and gate runs identically.

The replay adopts the published `RUN_ID` and asserts the traces cover the item set and budget grid
this notebook defines. A file that does not match is refused rather than plotted.


In [5]:
results, model_meta, SWEPT = {}, {}, []

if RUN_MODE == "REPLAY":
    if TRACES_URL and not TRACES_PATH.exists():
        print(f"fetching published generations from {TRACES_URL}")
        TRACES_PATH.write_text(requests.get(TRACES_URL, timeout=180).text, encoding="utf-8")
    assert TRACES_PATH.exists(), (
        f"REPLAY needs published generations at {TRACES_PATH} and found none. Set TRACES_URL "
        f"in the knobs cell, or use RUN_MODE = 'QUICK' to generate your own.")

    raw = [json.loads(l) for l in TRACES_PATH.open(encoding="utf-8") if l.strip()]
    assert raw, f"{TRACES_PATH} is empty"
    RUN_ID = Counter(r["run_id"] for r in raw).most_common(1)[0][0]
    raw = [r for r in raw if r["run_id"] == RUN_ID]

    want = {r["unique_id"] for r in items}
    missing = want - {r["problem_id"] for r in raw}
    assert not missing, (
        f"the published traces miss {len(missing)} of this notebook's {len(want)} problems; "
        f"that file was made with a different seed, dataset or item count.")
    have_budgets = sorted({r["budget"] for r in raw if r.get("budget") is not None})
    assert set(BUDGETS) <= set(have_budgets), (
        f"the published traces cover budgets {have_budgets}, not this notebook's {BUDGETS}.")

    # Only the roster's checkpoints. The section 9 inset writes its arm under a
    # pseudo model_id of "soft", which is not a curve and has no parameter count.
    roster = {m for m, _ in MODELS}
    SOFT = [r for r in raw if r["model_id"] not in roster and r.get("arm") != "wait"]
    # Section 11's arm keys on `waits` rather than on a budget and is read back whole.
    WAIT_RAW = [r for r in raw if r.get("arm") == "wait" and r["problem_id"] in want
                and r["model_id"] in roster and r["waits"] in set(WAITS)]
    for r in raw:
        if r["model_id"] in roster and r["problem_id"] in want and r["budget"] in set(BUDGETS):
            results.setdefault(r["model_id"], {}).setdefault(r["budget"], []).append(r)
    assert results, (
        f"no records for any of {sorted(roster)}; the published traces are from a "
        f"different roster than this notebook's.")
    for mid, by_b in results.items():
        for b in by_b:
            by_b[b] = sorted(by_b[b], key=lambda r: r["problem_id"])
        label = mid.split("/")[-1]
        size = re.search(r"(\d+(?:\.\d+)?)B", label)
        model_meta[mid] = dict(id=mid, label=label,
                               params=int(float(size.group(1)) * 1e9) if size else None,
                               quant=None, dtype=None, tokenizer_sha256=None,
                               revision=None, replayed=True)

    SWEPT = [m for m, _ in MODELS if m in results] or sorted(results)
    n = sum(len(v) for by in results.values() for v in by.values())
    print(f"replaying {RUN_ID}")
    print(f"  {n:,} generations, {len(SWEPT)} model(s), {len(want)} items, nothing loaded")
    for mid in SWEPT:
        print(f"    {mid.split('/')[-1]:<24}" +
              "  ".join(f"{b}={len(results[mid].get(b, []))}" for b in BUDGETS))
else:
    WAIT_RAW = []
    print(f"{RUN_MODE}: generations are made below")


AssertionError: REPLAY needs published generations at out/page_06/traces.jsonl and found none. Set TRACES_URL in the knobs cell, or use RUN_MODE = 'QUICK' to generate your own.

In [ ]:
if RUN_MODE != "REPLAY":
    device = get_device()
    DTYPE = harness.pick_dtype(device)


    def load(model_id, quant):
        got = harness.load_model(model_id, device=device, dtype=DTYPE,
                                 load_in_8bit=(quant == "8bit"), warmup_tokens=16)
        return got["model"], got["tokenizer"], got["params"], got["tok_per_s"]


    model, tokenizer, N_PARAMS, WARMUP_TOK_S = load(*MODELS[0])
    free_memory = harness.free_memory       # takes nothing: the CALLER must del its model
else:
    device = DTYPE = model = tokenizer = None
    N_PARAMS = WARMUP_TOK_S = None
    free_memory = harness.free_memory
    print("REPLAY: no model loaded, no weights downloaded")

## 4. Where the cut lands

These checkpoints have no separate think block: the working and the answer are one stream, so
capping the stream caps the reasoning directly. That is simpler than the hybrid-model version of
this experiment, not weaker — there is no second channel for reasoning to hide in.

Two properties the intervention depends on, asserted rather than assumed: the end-of-turn ids are
known, so "it stopped on its own" is distinguishable from "we stopped it"; and the forced suffix
survives a tokenise round trip, so figure 1 draws the cut where it actually happened.

In [ ]:
if RUN_MODE != "REPLAY":
    demo_msgs = [{"role": "user", "content": render_prompt("What is 2+2?")}]
    chat = tokenizer.apply_chat_template(demo_msgs, tokenize=False, add_generation_prompt=True)

    print("the prompt as sent, ending where the model takes over:")
    print("-" * 74)
    print(chat)
    print("-" * 74)

    assert "<think>" not in chat, (
        "this checkpoint pre-fills a think block, so it is a hybrid thinking model and the "
        "intervention below is cutting the wrong stream")

    # Only the ids `generate` actually stops on. A wrong guess here turns a cut trace into
    # a finished one, which then propagates through the carry-forward and fabricates the
    # top of the curve.
    gen_eos = model.generation_config.eos_token_id
    EOS_IDS = set(gen_eos) if isinstance(gen_eos, (list, tuple)) else {gen_eos}
    EOS_IDS.discard(None)
    print(f"\nend-of-turn ids: {sorted(EOS_IDS)}")
else:
    EOS_IDS = None
    print("REPLAY: the chat template and stop ids belong to the generating run;")
    print("        the manifest below records them.")

## 5. The intervention: cap, force, grade

1. **Cap** — generate at most `budget` tokens. `max_new_tokens` *is* the dial.
2. **Force** — append `Final answer: \boxed{` and generate until that brace closes.
3. **Grade** — the inherited functions, unchanged.

**When the cap does not bind there is no intervention:** a model that finishes inside its budget is
recorded as a plain unforced generation, and section 10 reports how often that happened.

`tokens`, `cut_index` and `answer_start` are what make figure 1 drawable. The span between the last
two is our tokenisation of text the model never emitted, which is why the page tints it differently.

In [ ]:
# Shared plumbing; the reconstruction invariant is asserted in the harness.
tokenizer_fingerprint = harness.tokenizer_fingerprint
incremental_token_strings = harness.incremental_token_strings


def cut_at_boxed_close(pieces):
    '''We prefilled one open brace with "\\boxed{". Walk the model's continuation
    until that brace closes, tracking depth so nested LaTeX like \\frac{1}{2}
    is not cut in half. Returns (kept token strings, did it close).'''
    depth, kept = 1, []
    for p in pieces:
        out = ""
        for ch in p:
            out += ch
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    kept.append(out)              # may end mid-token, on purpose
                    return kept, True
        kept.append(out)
    return kept, False


@torch.inference_mode()
def continue_ids(ids, mdl, tok, max_new_tokens):
    '''Greedy continuation from an explicit id sequence: the prefill that budget
    forcing needs. Returns the newly generated ids only.'''
    inp = torch.tensor([ids], device=mdl.device)
    out = mdl.generate(
        input_ids=inp,
        attention_mask=torch.ones_like(inp),
        max_new_tokens=max_new_tokens,
        # Qwen3 ships sampling defaults in generation_config, so do_sample=False is
        # not enough on its own -- the sampling parameters have to be cleared too.
        do_sample=False,
        temperature=None, top_p=None, top_k=None,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
    )
    return out[0][inp.shape[1]:].tolist()


In [ ]:
FORCE_STEM = "\n\nFinal answer: \\boxed{"      # the brace we open and it must close


def run_at_budget(question, budget, mdl, tok, allowance=ANSWER_ALLOWANCE):
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(chat, add_special_tokens=False).input_ids

    # --- 1. cap -------------------------------------------------------------
    gen_ids = continue_ids(prompt_ids, mdl, tok, budget)

    # By the count alone: generate returns fewer than the cap only if the model stopped
    # on its own. Testing the last token against an end-of-turn id would be a guess, and
    # a wrong one marks a cut trace as finished and fabricates the top of the curve.
    hit_cap = len(gen_ids) >= budget
    while gen_ids and gen_ids[-1] in EOS_IDS:
        gen_ids.pop()                             # end-of-turn is not visible content

    own_text = tok.decode(gen_ids, skip_special_tokens=True)
    own_pieces = incremental_token_strings(gen_ids, tok)
    assert "".join(own_pieces) == own_text, (
        "token strings do not reconstruct the decoded text; the page animates these token "
        "by token, so a mismatch would put a lie on the screen")

    if not hit_cap:
        return dict(trace=own_text, tokens=own_pieces,
                    own_tokens=len(gen_ids), answer_tokens=0,
                    tokens_generated=len(gen_ids),
                    truncated=False, forced=False, forced_suffix="",
                    answer_closed=None, cut_index=None, answer_start=None)

    # --- 2. force -----------------------------------------------------------
    suffix_ids = tok(FORCE_STEM, add_special_tokens=False).input_ids
    suffix_pieces = incremental_token_strings(suffix_ids, tok)
    assert "".join(suffix_pieces) == FORCE_STEM, (
        "the forced suffix does not survive a tokenise/detokenise round trip on this "
        "checkpoint; figure 1 would draw the intervention in the wrong place")

    ans_ids = continue_ids(prompt_ids + gen_ids + suffix_ids, mdl, tok, allowance)
    while ans_ids and ans_ids[-1] in EOS_IDS:
        ans_ids.pop()
    ans_pieces = incremental_token_strings(ans_ids, tok)
    assert "".join(ans_pieces) == tok.decode(ans_ids, skip_special_tokens=True)
    kept, closed = cut_at_boxed_close(ans_pieces)     # truncated at the closing brace

    tokens = own_pieces + suffix_pieces + kept
    return dict(trace="".join(tokens), tokens=tokens,
                own_tokens=len(gen_ids), answer_tokens=len(kept),
                tokens_generated=len(gen_ids) + len(kept),   # our suffix is not its cost
                truncated=True, forced=True, forced_suffix=FORCE_STEM,
                answer_closed=closed,
                cut_index=len(own_pieces),
                answer_start=len(own_pieces) + len(suffix_pieces))

In [ ]:
def render_trace(g, width=94, head=6, tail=3):
    '''Print a trace the way the page draws it: the model's own words, the point
    where the budget ran out, the suffix we injected, and what it committed to.'''
    own = "".join(g["tokens"][: g["cut_index"]]) if g["forced"] else g["trace"]
    lines = textwrap.wrap(own.strip().replace("\n", " "), width) or ["(nothing generated)"]
    if len(lines) > head + tail:
        lines = lines[:head] + [f"… [{len(lines) - head - tail} lines elided] …"] + lines[-tail:]
    out = [f"  {ln}" for ln in lines]
    if g["forced"]:
        # cut_index is a token index, so the cut usually lands mid-word.
        out[-1] += " ▌"
        out.append("  " + "┄" * (width // 2) + "  budget spent, block closed for it")
        forced_text = "".join(g["tokens"][g["answer_start"]:])
        out.append(f"  >> Final answer: \\boxed{{{forced_text}"
                   f"{'' if g['answer_closed'] else '   (never closed the brace)'}")
    return "\n".join(out)


if RUN_MODE != "REPLAY":
    demo = items[0]
    print(textwrap.fill("Q: " + demo["problem"], 96))
    print("ground truth:", demo["answer"], "\n")

    for b in (BUDGETS[0], BUDGETS[len(BUDGETS) // 2], MAX_BUDGET):
        g = run_at_budget(demo["problem"], b, model, tokenizer)
        assert "".join(g["tokens"]) == g["trace"], "token strings do not reconstruct the text"
        got = extract_final_candidate(g["trace"])
        ok = bool(grade_answer(got, demo["answer"]))
        print("=" * 100)
        print(f"budget {b:>5}   thinking {g['own_tokens']:>4} tok   "
              f"forced={g['forced']}   answer={got!r}   {'CORRECT' if ok else 'wrong'}")
        print(render_trace(g))
else:
    _r = results[SWEPT[0]][BUDGETS[len(BUDGETS) // 2]][0]
    print(textwrap.fill("Q: " + _r["question"], 96))
    print("ground truth:", _r["ground_truth"], "\n")
    print(f"budget {_r['budget']:>5}   own {_r['own_tokens']:>4} tok   "
          f"forced={_r['forced']}   answer={_r['extracted_answer']!r}   "
          f"{'CORRECT' if _r['correct'] else 'wrong'}")
    print(render_trace(_r))

## 6. The sweep machinery

One pass per (model, budget) over the item set. The per-item record is the trace-bank schema
pages 7 to 9 share, with `budget` where page 5 has `rung`.

**Every budget is generated independently** rather than sliced out of one long run, because the
forced suffix changes what comes after it.

**One exception, and it is exact.** If the model stopped on its own at budget *b*, it walks the
identical greedy path at any larger budget, so those records carry forward with `carried_from`
recorded. Hence ascending budget order, and hence anything counting *generations* rather than
*records* must filter on `carried_from is null`.

In [ ]:
# In REPLAY the run id and the resume log belong to the published run, which the
# cell above already adopted; nothing here is generated.
if RUN_MODE != "REPLAY":
    # The run id keys the resume cache, so it names everything that changes the
    # numbers: without dtype and device, a CPU run and a later GPU run would merge
    # into one curve stitched from two numeric regimes.
    RUN_ID = (f"{RUN_MODE.lower()}-{DATASET}-n{len(items)}-{ITEM_SET_HASH[:8]}"
              f"-cap{MAX_BUDGET}-{str(DTYPE).rsplit('.', 1)[-1]}-{device.type}")


    STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("model_id", "budget", "problem_id"))
    STORE.report()
    DONE = STORE.done


    def run_budget(model_id, mdl, tok, budget, rows, label=None):
        '''One pass over the items at one cap. Ascending budget order is required for
        the carry-forward below to be sound.'''
        records, n_generated, n_carried = [], 0, 0

        bar = harness.progress(rows, desc=f"{label or f'budget {budget}':<22}", unit="item")
        for row in bar:
            key = (model_id, budget, row["unique_id"])
            if key in DONE:
                records.append(DONE[key])              # already on disk from an earlier run
                continue

            # Carry forward: an untruncated run at a smaller cap is bit-identical here.
            carried = None
            for smaller in [b for b in BUDGETS if b < budget][::-1]:
                prev = DONE.get((model_id, smaller, row["unique_id"]))
                if prev and not prev["truncated"]:
                    carried = prev
                    break

            if carried is not None:
                rec = dict(carried, budget=budget,
                           carried_from=carried.get("carried_from") or carried["budget"])
            else:
                g = run_at_budget(row["problem"], budget, mdl, tok)
                n_generated += 1
                extracted = extract_final_candidate(g["trace"])
                rec = {
                    "model_id": model_id,
                    "problem_id": row["unique_id"],
                    "level": int(row["level"]),
                    "subject": row["subject"],
                    "question": row["problem"],
                    "ground_truth": row["answer"],
                    "budget": budget,
                    "trace": g["trace"],
                    "tokens": g["tokens"],
                    "cut_index": g["cut_index"],
                    "answer_start": g["answer_start"],
                    "truncated": g["truncated"],
                    "forced": g["forced"],
                    "forced_suffix": g["forced_suffix"],
                    "think_closed": None,
                    "answer_closed": g["answer_closed"],
                    "extracted_answer": extracted,
                    "correct": bool(grade_answer(extracted, row["answer"])),
                    "own_tokens": g["own_tokens"],
                    "answer_tokens": g["answer_tokens"],
                    "tokens_generated": g["tokens_generated"],
                    "carried_from": None,
                }

            rec = STORE.append(rec)
            records.append(rec)
            if rec.get("carried_from") is not None:
                n_carried += 1
            # gen counts real generations; resumed and carried-forward records are free.
            bar.set_postfix(acc=f"{mean([r['correct'] for r in records]):.2f}",
                            gen=n_generated, carried=n_carried)

        return records

## 7. Pilot: does the grid still bracket this model?

Page 5 already established the headroom on these checkpoints, so the pilot's remaining job is
narrower and mechanical: confirm the **top** of the grid is not still binding. If the largest
budget truncates a large share of items, the curve's right end is a truncation rather than a
finished chain, and the grid needs extending.

These generations are the sweep's own, cached by `RUN_ID`, so the pilot is free.

In [ ]:
if RUN_MODE != "REPLAY":
    PILOT_N = min(30 if RUN_MODE == "FULL" else 10, len(items))
    pilot = run_budget(MODELS[0][0], model, tokenizer, MAX_BUDGET, items[:PILOT_N],
                       label=f"pilot @ {MAX_BUDGET}")

    pilot_acc = mean([r["correct"] for r in pilot])
    pilot_own = mean([r["own_tokens"] for r in pilot])
    pilot_trunc = mean([r["truncated"] for r in pilot])

    # The grid's top must not bind. Everything else about headroom page 5 already answered.
    if pilot_trunc > 0.10:
        verdict = "TOP OF GRID STILL BINDS"
        advice = (f"{pilot_trunc:.0%} of items are still truncated at {MAX_BUDGET} tokens, so the "
                  f"right end of the curve is a truncation rather than a finished chain. Add the "
                  f"next doubling to BUDGETS.")
    else:
        verdict, advice = "grid brackets the model", "proceed"

    print(f"\nat the top budget ({MAX_BUDGET})   accuracy {pilot_acc:.0%}   "
          f"mean own tokens {pilot_own:,.0f}   still truncated {pilot_trunc:.0%}")
    print(f"-> {verdict}\n   {advice}")

    PILOT = dict(n=PILOT_N, budget=MAX_BUDGET, accuracy=pilot_acc,
                 mean_own_tokens=pilot_own, truncation_rate=pilot_trunc,
                 verdict=verdict, advice=advice)
else:
    PILOT = None
    print("REPLAY: the pilot is a property of the generating run; see the manifest")

## 8. The full sweep

Ascending budgets, one model at a time, the second loaded only after the first is dropped.
Nothing is decided here; this produces the log that sections 10 to 15 read.

In [ ]:
if RUN_MODE != "REPLAY":
    _worst = len(items) * sum(BUDGETS)
    print(f"up to {_worst:,} tokens per model; at ~{WARMUP_TOK_S:.0f} tok/s that is under "
          f"{_worst / max(WARMUP_TOK_S, 1e-6) / 3600:.1f} h each, less whatever carries forward.\n")

    results, model_meta = {}, {}


    def sweep_model(model_id, mdl, tok, n_params, quant):
        results[model_id] = {}
        model_meta[model_id] = dict(id=model_id, label=model_id.split("/")[-1], params=n_params,
                                    quant=quant, dtype=str(next(mdl.parameters()).dtype),
                                    tokenizer_sha256=harness.tokenizer_fingerprint(tok),
                                    revision=harness.repo_revision(model_id))
        for b in BUDGETS:                                   # ascending: carry-forward needs it
            results[model_id][b] = run_budget(model_id, mdl, tok, b, items)
        return results[model_id]


    sweep_model(MODELS[0][0], model, tokenizer, N_PARAMS, MODELS[0][1])

    # Carry-forward claims an untruncated run at a smaller cap is bit-identical at a larger
    # one. Checked here rather than in section 10, because by then this model has been freed.
    CARRY_CHECK = None
    _c = next((r for b in BUDGETS for r in results[MODELS[0][0]][b] if r.get("carried_from")), None)
    if _c is not None:
        _again = run_at_budget(_c["question"], _c["budget"], model, tokenizer)
        CARRY_CHECK = dict(problem_id=_c["problem_id"], budget=_c["budget"],
                           carried_from=_c["carried_from"],
                           identical=_again["trace"] == _c["trace"])
        print(f"\ncarry-forward check: {_c['problem_id']} @ {_c['budget']} -> "
              f"{'identical' if CARRY_CHECK['identical'] else 'DIFFERENT'}")
        assert CARRY_CHECK["identical"], (
            "a carried-forward record does not reproduce when generated directly. Delete "
            "traces.jsonl and re-sweep before trusting anything below.")
    else:
        print("\ncarry-forward check: nothing was carried forward")
else:
    # The carry-forward spot check is a property of the generating run; the manifest
    # below records its verdict rather than re-deriving it without a model.
    CARRY_CHECK = None


## 9. Asked nicely versus cut off

Same items, one nominal budget, two ways of imposing it: **asked nicely**, where the prompt
requests brevity in words and generation runs to the top of the grid, and **cut off**, the hard
cap the rest of this notebook uses.

Read the token columns, not the accuracies. The soft arm was free to run to `MAX_BUDGET` and the
hard arm was capped, so their accuracies are not comparable.

In [ ]:
# The soft-request arm is a generation, so a replay reads it back instead. Its
# records are in the trace bank under a pseudo model id, which the replay cell
# separated out as SOFT.
if RUN_MODE != "REPLAY":
    SOFT_N = min(20, len(items))
    SOFT_BUDGET = BUDGETS[len(BUDGETS) // 2]         # a middle rung of the grid

    # A plain brevity instruction, not "use at most N tokens": a model cannot count its
    # own tokens as it emits them, so a numeric self-budget would be an unfair request.
    SOFT_REQUEST = "\n\nKeep your reasoning to one or two sentences, then give the final answer."


    def median(xs):
        xs = sorted(xs)
        m = len(xs) // 2
        return xs[m] if len(xs) % 2 else (xs[m - 1] + xs[m]) / 2


    soft = []
    for k, row in enumerate(items[:SOFT_N], start=1):
        key = ("soft", SOFT_BUDGET, row["unique_id"])
        if key in DONE:
            rec = DONE[key]
        else:
            g = run_at_budget(row["problem"] + SOFT_REQUEST, MAX_BUDGET, model, tokenizer)
            got = extract_final_candidate(g["trace"])
            rec = {"model_id": "soft", "problem_id": row["unique_id"],
                   "level": int(row["level"]), "subject": row["subject"],
                   "question": row["problem"], "ground_truth": row["answer"],
                   "budget": SOFT_BUDGET, "arm": "asked_nicely", "request": SOFT_REQUEST.strip(),
                   "trace": g["trace"], "tokens": g["tokens"],
                   "cut_index": g["cut_index"], "answer_start": g["answer_start"],
                   "truncated": g["truncated"], "forced": g["forced"],
                   "forced_suffix": g["forced_suffix"], "think_closed": None,
                   "answer_closed": g["answer_closed"], "extracted_answer": got,
                   "correct": bool(grade_answer(got, row["answer"])),
                   "own_tokens": g["own_tokens"], "answer_tokens": g["answer_tokens"],
                   "tokens_generated": g["tokens_generated"], "carried_from": None}
            rec = STORE.append(rec)      # the inset ships in the download tier too
        soft.append(rec)
        print(f"\rasked nicely: {k}/{SOFT_N}", end="")
    print()

    soft_ids = {r["problem_id"] for r in soft}
    hard_runs = [r for r in results[PRIMARY][SOFT_BUDGET] if r["problem_id"] in soft_ids]

    INSET = {
        "nominal_budget": SOFT_BUDGET,
        "n": SOFT_N,
        "request": SOFT_REQUEST.strip(),
        "soft": dict(mean_own_tokens=mean([r["own_tokens"] for r in soft]),
                     median_own_tokens=median([r["own_tokens"] for r in soft]),
                     over_budget_rate=mean([r["own_tokens"] > SOFT_BUDGET for r in soft]),
                     measured_at_budget=MAX_BUDGET,
                     accuracy=mean([r["correct"] for r in soft])),
        "hard": dict(mean_own_tokens=mean([r["own_tokens"] for r in hard_runs]),
                     median_own_tokens=median([r["own_tokens"] for r in hard_runs]),
                     over_budget_rate=0.0,
                     measured_at_budget=SOFT_BUDGET,
                     accuracy=mean([r["correct"] for r in hard_runs])),
    }
    # Against the NOMINAL budget, the number both arms were asked for.
    INSET["overshoot_multiple"] = INSET["soft"]["mean_own_tokens"] / SOFT_BUDGET

    print(f"\nnominal budget: {SOFT_BUDGET} tokens, n={SOFT_N}\n")
    print(f"{'':<14}{'mean tok':>10}{'median':>9}{'over budget':>13}{'accuracy':>10}")
    for k in ("soft", "hard"):
        d = INSET[k]
        print(f"{'asked nicely' if k == 'soft' else 'cut off':<14}"
              f"{d['mean_own_tokens']:>10,.0f}{d['median_own_tokens']:>9,.0f}"
              f"{d['over_budget_rate']:>12.0%}{d['accuracy']:>10.0%}")
    print(f"\nasked nicely runs {INSET['overshoot_multiple']:.1f}x the nominal {SOFT_BUDGET}-token "
          f"budget; the cap runs 1.0x by construction")
    print("the two accuracy figures are NOT comparable: the soft arm was free to run to "
          f"{MAX_BUDGET}\ntokens, the hard arm was capped at {SOFT_BUDGET}. The columns to read are the "
          "token columns.")
else:
    soft = SOFT
    if soft:
        SOFT_BUDGET = soft[0]["budget"]
        soft_ids = {r["problem_id"] for r in soft}
        hard_runs = [r for r in results[SWEPT[0]][SOFT_BUDGET]
                     if r["problem_id"] in soft_ids]
        INSET = {"nominal_budget": SOFT_BUDGET, "n": len(soft),
                 "request": soft[0].get("request"),
                 "soft": dict(mean_own_tokens=mean([r["own_tokens"] for r in soft]),
                              median_own_tokens=median([r["own_tokens"] for r in soft]),
                              over_budget_rate=mean([r["own_tokens"] > SOFT_BUDGET for r in soft]),
                              accuracy=mean([r["correct"] for r in soft])),
                 "hard": dict(mean_own_tokens=mean([r["own_tokens"] for r in hard_runs]),
                              median_own_tokens=median([r["own_tokens"] for r in hard_runs]),
                              over_budget_rate=0.0,
                              accuracy=mean([r["correct"] for r in hard_runs]))}
        INSET["overshoot_multiple"] = INSET["soft"]["mean_own_tokens"] / SOFT_BUDGET
        print(f"replayed inset: asked nicely runs "
              f"{INSET['overshoot_multiple']:.1f}x the nominal {SOFT_BUDGET}-token budget")
    else:
        INSET = None
        print("no soft-request records in the published traces; figure 1 has no inset")

In [ ]:
if RUN_MODE != "REPLAY":
    # The other curves, one resident at a time: peak memory is a single checkpoint.
    if len(MODELS) > 1:
        del model
        free_memory()
        for mid, quant in MODELS[1:]:
            try:
                mdl, tok, n_p, _ = load(mid, quant)
                sweep_model(mid, mdl, tok, n_p, quant)
                del mdl
                free_memory()
            except Exception as e:      # OOM is the expected one; any load failure survivable
                print(f"\n!! {mid} did not load ({type(e).__name__}). Figure 2 will be drawn "
                      f"from the models that did.\n   {str(e)[:160]}")
                free_memory()

    SWEPT = [m for m, _ in MODELS if m in results]
    print("\nswept:", ", ".join(m.split("/")[-1] for m in SWEPT))

## 10. Sweep integrity

Six checks, four in the next cell and two in the one after. Two of them are specific to this page:

- **Check 3** — every trace at budget *b* holds at most *b* tokens of the model's own thinking.
  This is the assertion that the x-axis is what it claims.
- **Check 5** — a trace at a small budget is a genuine prefix of the trace at a large one. Figure
  1 greys out what the model *would* have written past the cut, and that is only honest if the
  visible text really is the beginning of it. It also licenses the carry-forward in section 6.

In [ ]:
all_records = [r for mid in SWEPT for b in BUDGETS for r in results[mid][b]]
forced_records = [r for r in all_records if r["forced"]]

print(f"{len(all_records):,} records  ({len(forced_records):,} forced)\n")

# 1. every trace reconstructs from its token strings
bad = [r for r in all_records if "".join(r["tokens"]) != r["trace"]]
print(f"1. traces reconstruct from tokens ......... {len(bad)} violations")
assert not bad, "a trace's token strings do not join back to its text; figure 1's reveal would lie"

# 2. the forced answer had room to close its brace
unclosed = [r for r in forced_records if r["answer_closed"] is False]
rate = len(unclosed) / max(len(forced_records), 1)
print(f"2. forced answers closed the brace ........ {1 - rate:.1%} "
      f"({len(unclosed)} of {len(forced_records)} ran out of allowance)")
if rate > 0.05:
    print(f"   ^ above 5%: raise ANSWER_ALLOWANCE (now {ANSWER_ALLOWANCE}) and re-run. An unclosed "
          f"brace is not\n     merely graded wrong -- get_last_boxed returns nothing, so "
          f"extract_final_candidate falls\n     back to the last NUMBER anywhere in the trace, "
          f"which can score right or wrong by luck.\n     Those items are graded by a different "
          f"rule than the rest, which is ours, not the model's.")

# 3. the cap actually bound where it says it did
over = [r for r in all_records if r["own_tokens"] > r["budget"]]
print(f"3. thinking never exceeded its budget ..... {len(over)} violations")
assert not over, "a trace thought for longer than its cap; the x-axis does not mean what it says"

# 4. the dial bites at the bottom of the grid and lets go at the top
print("\n4. how often the cap bound, by budget:")
print(f"   {'budget':>8}" + "".join(f"{mid.split('/')[-1]:>16}" for mid in SWEPT))
for b in BUDGETS:
    row = "".join(f"{mean([r['truncated'] for r in results[mid][b]]):>15.0%} " for mid in SWEPT)
    print(f"   {b:>8}" + row)
print("   (falling to zero at the top of the grid means the largest budget is no longer a")
print("    constraint -- past that point the page's dial is not doing anything)")


In [ ]:
# 5. a small-budget trace is a genuine prefix of a large-budget one
def prefix_len(a, b):
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


prefix_report = []
for mid in SWEPT:
    ref = {r["problem_id"]: r for r in results[mid][MAX_BUDGET]}
    for b in BUDGETS[:-1]:
        for r in results[mid][b]:
            R = ref[r["problem_id"]]
            own = r["tokens"][: r["cut_index"]] if r["forced"] else r["tokens"]
            ref_own = R["tokens"][: R["cut_index"]] if R["forced"] else R["tokens"]
            n = prefix_len(own, ref_own)
            prefix_report.append(dict(model_id=mid, budget=b, problem_id=r["problem_id"],
                                      shared=n, own=len(own), exact=n == len(own),
                                      # only a truncated record is an independent generation
                                      independent=r["truncated"]))

indep = [p for p in prefix_report if p["independent"]]
exact = mean([p["exact"] for p in indep])
mismatches = sum(not p["exact"] for p in indep)
print(f"5. short traces are prefixes of long ones . {exact:.1%} exact "
      f"({mismatches} of {len(indep)} diverge)")
print(f"   {len(prefix_report) - len(indep)} of {len(prefix_report)} comparisons excluded as "
      f"self-comparisons (the\n   record at the larger budget IS the record at the smaller one, "
      f"carried forward)")
if not indep:
    print("   nothing independent to compare: no record was truncated below the top budget")
elif exact < 1.0:
    worst = min(prefix_report, key=lambda p: p["shared"] / max(p["own"], 1))
    print(f"   worst case shares {worst['shared']}/{worst['own']} tokens "
          f"({worst['model_id']} @ {worst['budget']}, {worst['problem_id']})")
    print("   Some divergence is expected: greedy argmax is only deterministic up to floating-point")
    print("   reduction order, which shifts with sequence length on some kernels. A few near-total")
    print("   prefixes are fine. Widespread short ones mean do_sample slipped back on -- check that")
    print("   first, because everything downstream assumes the dial is the only variable.")

# 6. carry-forward is exact, not an approximation -- re-run one and compare
carried = [r for r in all_records if r.get("carried_from")]
print(f"\n6. records carried forward from a smaller budget: {len(carried)} of {len(all_records)}")
if CARRY_CHECK is None:
    print("   no spot check: nothing was carried forward on the primary model")
else:
    print(f"   spot check (run in section 8, while the primary was still loaded): "
          f"{CARRY_CHECK['problem_id']} @ {CARRY_CHECK['budget']} "
          f"-> {'identical' if CARRY_CHECK['identical'] else 'DIFFERENT'}")

INTEGRITY = dict(n_records=len(all_records), n_forced=len(forced_records),
                 unclosed_forced_rate=rate, prefix_exact_rate=exact,
                 n_prefix_comparisons=len(indep), n_carried_forward=len(carried),
                 carry_forward_spot_check=CARRY_CHECK)


## 11. The other direction: told to keep going

Everything above takes room away. This refuses to let the model stop: when it tries to end its
turn, the end-of-turn token is dropped, the word `Wait` is appended to what it has already
written, and generation continues from there. Repeat *k* times, grade after each.

That is the extension half of budget forcing, and the sweep above was the terminating half. The
two arms share the prompt, the items, the decoding and the grader, so they differ in one thing.

**These checkpoints have no thinking block, so the injection lands after a finished answer, not
inside a deliberation.** s1 injects before the model commits; here there is nothing to inject
into but the end of the turn. What that difference is allowed to support is
[`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §7.

Worth naming before the numbers arrive: s1-32B is Qwen2.5-32B-Instruct **after** supervised
fine-tuning on a thousand reasoning traces. These are the same family without that step, so this
arm is the ablation that paper leaves implicit — the word, without the training.


In [ ]:
WAIT_TEXT = "\n" + WAIT_WORD          # the injection, exactly as it lands in the stream


def run_with_waits(question, mdl, tok, n_waits,
                   allowance=WAIT_ALLOWANCE, base_allowance=None):
    """Generate; each time the model tries to end its turn, drop the end-of-turn token,
    append the word and let it carry on.

    Returns one record per k in 0..n_waits. Record k IS record k-1 plus our word plus
    whatever followed it, so this is one growing stream sampled k+1 times rather than
    k+1 separate runs -- which is what makes "watch it think longer" an honest thing
    for the page to draw.

    k=0 gets `base_allowance`, which is the sweep's top budget, because k=0 IS the
    model left alone and a tighter cap there would make it a cap experiment again.
    Every injection after it buys `allowance` more.
    """
    base_allowance = MAX_BUDGET if base_allowance is None else base_allowance
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(chat, add_special_tokens=False).input_ids

    wait_ids = tok(WAIT_TEXT, add_special_tokens=False).input_ids
    wait_pieces = incremental_token_strings(wait_ids, tok)
    assert "".join(wait_pieces) == WAIT_TEXT, (
        "the injected word does not survive a tokenise round trip on this checkpoint; the "
        "page tints those tokens differently and would tint the wrong ones")
    suffix_ids = tok(FORCE_STEM, add_special_tokens=False).input_ids
    suffix_pieces = incremental_token_strings(suffix_ids, tok)

    body, spans, out, k = [], [], [], 0
    while True:
        room = base_allowance if k == 0 else allowance
        fresh = continue_ids(prompt_ids + body, mdl, tok, room)
        ran_on = len(fresh) >= room               # it never offered a stop to suppress
        while fresh and fresh[-1] in EOS_IDS:
            fresh.pop()
        body += fresh

        pieces = incremental_token_strings(body, tok)
        text = tok.decode(body, skip_special_tokens=True)
        assert "".join(pieces) == text, (
            "token strings do not reconstruct the decoded text; the page reveals these token "
            "by token, so a mismatch would put a lie on the screen")
        ours = sum(e - s for s, e in spans)       # the words we injected are not its cost

        if ran_on:
            # Still going when we stopped it, so force the answer exactly as the cap arm
            # does. Without this the record grades as "no answer" for a reason of ours,
            # and the two arms would not be graded alike.
            ans = continue_ids(prompt_ids + body + suffix_ids, mdl, tok, ANSWER_ALLOWANCE)
            while ans and ans[-1] in EOS_IDS:
                ans.pop()
            kept, closed = cut_at_boxed_close(incremental_token_strings(ans, tok))
            toks = pieces + suffix_pieces + kept
            rec = dict(trace="".join(toks), tokens=toks,
                       cut_index=len(pieces), answer_start=len(pieces) + len(suffix_pieces),
                       truncated=True, forced=True, forced_suffix=FORCE_STEM,
                       answer_closed=closed, answer_tokens=len(kept),
                       own_tokens=len(body) - ours,
                       tokens_generated=len(body) - ours + len(kept))
        else:
            rec = dict(trace=text, tokens=pieces,
                       cut_index=None, answer_start=None,
                       truncated=False, forced=False, forced_suffix="",
                       answer_closed=None, answer_tokens=0,
                       own_tokens=len(body) - ours, tokens_generated=len(body) - ours)

        rec.update(waits=k, wait_spans=[list(s) for s in spans], offered_a_stop=not ran_on)
        out.append(rec)

        if k >= n_waits or ran_on:
            break
        spans.append((len(body), len(body) + len(wait_ids)))
        body += wait_ids
        k += 1
    return out


In [ ]:
# The injection loop, against a fake model. REPLAY never generates, so nothing else
# in this notebook can catch a bug in it; this runs in every mode and costs nothing.
# The tokenizer is char-level, so decoding is exact and the reconstruction assert
# above runs for real rather than against a stub that agrees with itself.
def _wait_selftest():
    EOS, RAN_ON = 0, object()

    class Tok:
        pad_token_id, eos_token_id = None, EOS
        def apply_chat_template(self, msgs, **kw): return "P:" + msgs[0]["content"]
        def __call__(self, text, **kw):
            return type("E", (), {"input_ids": [ord(c) for c in text]})()
        def decode(self, ids, **kw): return "".join(chr(i) for i in ids if i != EOS)

    def wire(script):
        calls = []
        def fake_continue(ids, mdl, tok, max_new_tokens):
            step = script[len(calls)]
            calls.append(max_new_tokens)
            if step is RAN_ON:
                return [ord("x")] * max_new_tokens      # never offered a stop
            return [ord(c) for c in step] + [EOS]
        return calls, fake_continue

    real_continue, real_pieces = continue_ids, incremental_token_strings
    globals()["incremental_token_strings"] = lambda ids, tok: [chr(i) for i in ids]
    try:
        # A. it offers a stop every time: k+1 records, each extending the one below.
        calls, globals()["continue_ids"] = wire(["one.", "two.", "three.", "four."])
        recs = run_with_waits("Q", None, Tok(), 3)
        assert [r["waits"] for r in recs] == [0, 1, 2, 3]
        assert not any(r["forced"] for r in recs)
        for a, b in zip(recs, recs[1:]):
            assert b["trace"].startswith(a["trace"])
        assert recs[1]["trace"] == "one.\nWaittwo."
        assert all(recs[3]["trace"][s:e] == WAIT_TEXT for s, e in recs[3]["wait_spans"])
        assert recs[3]["own_tokens"] == len(recs[3]["trace"]) - 3 * len(WAIT_TEXT)
        assert calls[0] == MAX_BUDGET and set(calls[1:]) == {WAIT_ALLOWANCE}

        # B. it runs on: nothing left to suppress, so we stop and force the answer.
        calls, globals()["continue_ids"] = wire(["one.", RAN_ON, "42}"])
        recs = run_with_waits("Q", None, Tok(), 3)
        assert [r["waits"] for r in recs] == [0, 1]
        assert recs[1]["forced"] and recs[1]["answer_closed"] is True
        assert recs[1]["trace"].endswith(FORCE_STEM + "42}")
        assert "".join(recs[1]["tokens"][recs[1]["answer_start"]:]) == "42}"
        assert recs[1]["own_tokens"] == len("one.") + WAIT_ALLOWANCE   # our stem is not its cost
        assert recs[1]["offered_a_stop"] is False
    finally:
        globals()["continue_ids"], globals()["incremental_token_strings"] = real_continue, real_pieces
    return len(recs)


_wait_selftest()
print("wait-arm generation path: records extend, our words are not its cost, "
      "a run-on is forced")


In [ ]:
WAIT_RESULTS = {}

if RUN_MODE != "REPLAY":
    # This arm keys on `waits`, which the budget arm's records do not carry, so its
    # resume map is read from the file rather than from STORE.done.
    WAIT_DONE = {}
    if TRACES_PATH.exists():
        for _line in TRACES_PATH.open(encoding="utf-8"):
            if _line.strip():
                _r = json.loads(_line)
                if _r.get("run_id") == RUN_ID and _r.get("arm") == "wait":
                    WAIT_DONE[(_r["model_id"], _r["waits"], _r["problem_id"])] = _r


    def run_wait_pass(model_id, mdl, tok, rows):
        """One pass over the items, k = 0 to max(WAITS), for one checkpoint."""
        records, n_generated = [], 0
        bar = harness.progress(rows, desc=f"{'waits ' + model_id.split('/')[-1]:<22}",
                               unit="item")
        for row in bar:
            cached = [WAIT_DONE.get((model_id, k, row["unique_id"])) for k in WAITS]
            if all(c is not None for c in cached):
                records += cached
                continue

            snaps = {s["waits"]: s
                     for s in run_with_waits(row["problem"], mdl, tok, max(WAITS))}
            n_generated += 1
            last = None
            for k in WAITS:
                s = snaps.get(k)
                carried = None
                if s is None:
                    # It stopped offering stops, so there was nothing left to suppress
                    # and the state at this k is the state at the last one.
                    s, carried = last, last["waits"]
                got = extract_final_candidate(s["trace"])
                rec = {
                    "model_id": model_id,
                    "problem_id": row["unique_id"],
                    "level": int(row["level"]),
                    "subject": row["subject"],
                    "question": row["problem"],
                    "ground_truth": row["answer"],
                    "arm": "wait",
                    "budget": None,
                    "waits": k,
                    "wait_word": WAIT_WORD,
                    "trace": s["trace"],
                    "tokens": s["tokens"],
                    "wait_spans": s["wait_spans"],
                    "cut_index": s["cut_index"],
                    "answer_start": s["answer_start"],
                    "truncated": s["truncated"],
                    "forced": s["forced"],
                    "forced_suffix": s["forced_suffix"],
                    "think_closed": None,
                    "answer_closed": s["answer_closed"],
                    "offered_a_stop": s["offered_a_stop"],
                    "extracted_answer": got,
                    "correct": bool(grade_answer(got, row["answer"])),
                    "own_tokens": s["own_tokens"],
                    "answer_tokens": s["answer_tokens"],
                    "tokens_generated": s["tokens_generated"],
                    "carried_from": carried,
                }
                rec = STORE.append(rec)
                WAIT_DONE[(model_id, k, row["unique_id"])] = rec
                records.append(rec)
                if snaps.get(k) is not None:
                    last = s
            at_zero = [r for r in records if r["waits"] == 0]
            at_top = [r for r in records if r["waits"] == max(WAITS)]
            bar.set_postfix(k0=f"{mean([r['correct'] for r in at_zero]):.2f}",
                            kmax=f"{mean([r['correct'] for r in at_top]):.2f}",
                            gen=n_generated)
        return records


    # Priced from what the sweep above actually spent rather than from the caps: at
    # the top budget the primary wrote a mean of MEAN_TOP tokens, and each injection
    # buys at most WAIT_ALLOWANCE more.
    _mean_top = mean([r["own_tokens"] for r in results[PRIMARY][MAX_BUDGET]])
    _likely = len(items) * (_mean_top + max(WAITS) * WAIT_ALLOWANCE * 0.6)
    _worst = len(items) * (MAX_BUDGET + max(WAITS) * WAIT_ALLOWANCE)
    print(f"k=0 alone costs about {len(items) * _mean_top:,.0f} tokens per model, the same "
          f"chains the sweep\n  already wrote at {MAX_BUDGET}; it is regenerated so that "
          f"the two arms are one stream.")
    print(f"expect roughly {_likely:,.0f} tokens per model, at most {_worst:,} .\n")

    for _mid, _quant in [(m, q) for m, q in MODELS if m in results]:
        _mdl, _tok, _n, _t = load(_mid, _quant)
        WAIT_RESULTS[_mid] = run_wait_pass(_mid, _mdl, _tok, items)
        del _mdl
        free_memory()

    print("\nswept with waits:", ", ".join(m.split("/")[-1] for m in WAIT_RESULTS))
else:
    for _mid in SWEPT:
        rs = [r for r in WAIT_RAW if r["model_id"] == _mid]
        if rs:
            WAIT_RESULTS[_mid] = sorted(rs, key=lambda r: (r["waits"], r["problem_id"]))
    print("REPLAY:", sum(len(v) for v in WAIT_RESULTS.values()), "wait-arm records read back")


In [ ]:
# Four checks, and the first is the one that ties this arm to the one above.
WAIT_CHECKS = {}

if WAIT_RESULTS:
    _p = PRIMARY if PRIMARY in WAIT_RESULTS else sorted(WAIT_RESULTS)[0]
    flat = [r for rs in WAIT_RESULTS.values() for r in rs]

    # 1. k=0 is the model left alone, which is exactly the cap arm at the top budget
    #    whenever that run was not truncated. Greedy decoding is deterministic, so a
    #    disagreement means the two arms are not the same experiment.
    ref = {r["problem_id"]: r for r in results[_p][MAX_BUDGET] if not r["truncated"]}
    zero = [r for r in WAIT_RESULTS[_p] if r["waits"] == 0 and r["problem_id"] in ref]
    same = sum(r["trace"] == ref[r["problem_id"]]["trace"] for r in zero)
    WAIT_CHECKS["k0_matches_uncapped"] = (same, len(zero))
    print(f"1. k=0 reproduces the uncapped run ..... {same}/{len(zero)} traces identical")
    if zero and same < len(zero):
        print("   Greedy argmax is deterministic only up to floating-point reduction order, so")
        print("   a few may differ. Widespread disagreement means one arm is not greedy.")

    # 2. every trace still rebuilds from its token strings
    bad = [r for r in flat if "".join(r["tokens"]) != r["trace"]]
    print(f"2. traces reconstruct from tokens ...... {len(bad)} violations")
    assert not bad, "a wait-arm trace's token strings do not join back to its text"

    # 3. record k is a genuine prefix of record k+1. True by construction -- one stream,
    #    sampled -- and asserted because the page draws the growth as continuous.
    breaks = 0
    for mid, rs in WAIT_RESULTS.items():
        by_pid = {}
        for r in rs:
            by_pid.setdefault(r["problem_id"], {})[r["waits"]] = r
        for pid, by_k in by_pid.items():
            for a, b in zip(WAITS, WAITS[1:]):
                if a in by_k and b in by_k:
                    own_a = "".join(by_k[a]["tokens"][: by_k[a]["cut_index"]]) \
                        if by_k[a]["forced"] else by_k[a]["trace"]
                    own_b = "".join(by_k[b]["tokens"][: by_k[b]["cut_index"]]) \
                        if by_k[b]["forced"] else by_k[b]["trace"]
                    if not own_b.startswith(own_a):
                        breaks += 1
    WAIT_CHECKS["prefix_breaks"] = breaks
    print(f"3. each k extends the k before it ...... {breaks} violations")
    assert not breaks, "a wait-arm record is not an extension of the one below it"

    # 4. the injected words are not charged to the model
    charged = [r for r in flat
               if r["own_tokens"] != len(r["tokens"][: r["cut_index"]] if r["forced"]
                                         else r["tokens"]) - sum(e - s for s, e in r["wait_spans"])]
    WAIT_CHECKS["miscounted"] = len(charged)
    print(f"4. our words are not its token cost .... {len(charged)} violations")
    assert not charged, "an injected word was counted as the model's own output"

    # How often there was a stop to suppress at all: a model that never offers one
    # cannot be extended, and its later k are carried rather than generated.
    print("\n   how often the model offered a stop, by k:")
    print(f"   {'k':>4}" + "".join(f"{m.split('/')[-1]:>18}" for m in WAIT_RESULTS))
    for k in WAITS:
        row = "".join(f"{mean([r['offered_a_stop'] for r in rs if r['waits'] == k]):>17.0%} "
                      for rs in WAIT_RESULTS.values())
        print(f"   {k:>4}" + row)
else:
    print("no wait-arm records; section 11 did not run")


# Part 2 — What the page shows

Every figure, number and gate the page renders, derived from what part 1 produced and written to
`out/page_06/`. No generation happens here, so these cells are cheap to re-run while a figure is being
adjusted.

## 12. Figure 1 — the truncation view

One problem, every budget, the trace cut where the reader put the dial.

The specimen is **curated from the sweep, not generated for the figure**: wrong at a low budget,
right at a high one, flip near the middle of the grid, and the shortest traces among those.

`reference_tokens` is the full trace the page greys out past the cut, `answer_start` is where the
tint changes, and the per-budget `correct` flags are what make the verdict chip flip
discontinuously.

In [ ]:
def by_problem(mid):
    '''problem_id -> {budget: record}'''
    out = {}
    for b in BUDGETS:
        for r in results[mid][b]:
            out.setdefault(r["problem_id"], {})[b] = r
    return out


PRIMARY_BY_PROBLEM = by_problem(PRIMARY)


def first_correct(row):
    for b in BUDGETS:
        if row[b]["correct"]:
            return b
    return None


candidates = []
for pid, row in PRIMARY_BY_PROBLEM.items():
    flip = first_correct(row)
    if flip is None or flip == BUDGETS[0]:
        continue                                   # need a visible "before"
    if not row[BUDGETS[BUDGETS.index(flip) - 1]]["truncated"]:
        continue                                   # the budget below must show a real cut
    centre = abs(BUDGETS.index(flip) - (len(BUDGETS) - 1) / 2)
    candidates.append((centre, len(row[MAX_BUDGET]["tokens"]), pid, flip))

candidates.sort()
print(f"{len(candidates)} problems flip above the first budget with a visible cut below it")

SPECIMEN_IS_FALLBACK = False
if candidates:
    _, _, SPECIMEN_ID, SPECIMEN_FLIP = candidates[0]
else:
    # Degrade with a flag rather than fail: figures 2 and 3 do not depend on the
    # specimen. A flagged specimen must not be translated to the page.
    SPECIMEN_IS_FALLBACK = True
    ranked = sorted(
        PRIMARY_BY_PROBLEM.items(),
        key=lambda kv: (first_correct(kv[1]) is None,          # prefer one that flips at all
                        not kv[1][BUDGETS[0]]["truncated"],    # then one with a visible cut
                        len(kv[1][MAX_BUDGET]["tokens"]),
                        kv[0]))
    SPECIMEN_ID = ranked[0][0]
    SPECIMEN_FLIP = first_correct(ranked[0][1])
    print("\nWARNING: no problem is wrong at a low budget and right at a high one with a cut\n"
          "         below the flip. Every problem is solved at the smallest budget, or at\n"
          "         none. That is the pilot's too-easy / too-hard verdict reappearing.\n"
          "         Falling back to an illustrative specimen; derived.json flags it, and it\n"
          "         must not be translated to the page. Figures 2 and 3 are unaffected.\n")
specimen = PRIMARY_BY_PROBLEM[SPECIMEN_ID]
print(f"specimen: {SPECIMEN_ID}  (level {specimen[MAX_BUDGET]['level']}, flips at {SPECIMEN_FLIP})")


In [ ]:
print(textwrap.fill("Q: " + specimen[MAX_BUDGET]["question"], 96))
print("ground truth:", specimen[MAX_BUDGET]["ground_truth"])

for b in BUDGETS:
    r = specimen[b]
    print("\n" + "=" * 100)
    print(f"budget {b:>5}   thinking {r['own_tokens']:>4} tok   forced={r['forced']!s:<5}  "
          f"answer={r['extracted_answer']!r}   {'CORRECT' if r['correct'] else 'wrong'}"
          f"{'   <- flips here' if b == SPECIMEN_FLIP else ''}")
    print(render_trace(r))

verdicts = "".join("C" if specimen[b]["correct"] else "." for b in BUDGETS)
print(f"\nverdict across the grid: {verdicts}   (a step, not a ramp -- that is beat 1's whole point)")


## 13. Figure 2 — the population curve

Share of problems solved against token budget, log x-axis, one line per model.

**If the curve is not monotone, keep it.** The cell after the plot checks for a dip so it cannot
be smoothed away by accident.

In [ ]:
def curve_for(mid):
    pts = []
    for b in BUDGETS:
        rs = results[mid][b]
        pts.append(dict(
            budget=b,
            n=len(rs),
            n_correct=sum(r["correct"] for r in rs),
            accuracy=mean([r["correct"] for r in rs]),
            mean_tokens=mean([r["tokens_generated"] for r in rs]),
            total_tokens=sum(r["tokens_generated"] for r in rs),
            truncation_rate=mean([r["truncated"] for r in rs]),
            forced_rate=mean([r["forced"] for r in rs]),
        ))
    return pts


CURVES = {mid: curve_for(mid) for mid in SWEPT}

for mid in SWEPT:
    print(f"\n{mid}   n={CURVES[mid][0]['n']} items, MATH-500 levels 4-5, greedy")
    print(f"  {'budget':>8}{'solved':>10}{'accuracy':>10}{'mean tok':>10}{'total tok':>12}{'cap bound':>11}")
    for p in CURVES[mid]:
        print(f"  {p['budget']:>8}{p['n_correct']:>6}/{p['n']:<3}{p['accuracy']:>9.1%}"
              f"{p['mean_tokens']:>10,.0f}{p['total_tokens']:>12,}{p['truncation_rate']:>11.0%}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
COLOURS = ["#3b4a6b", "#a2704a"]

for (mid, pts), colour in zip(CURVES.items(), COLOURS):
    ax.plot([p["budget"] for p in pts], [p["accuracy"] * 100 for p in pts],
            marker="o", ms=4.5, lw=2, color=colour, label=model_meta[mid]["label"])

ax.set_xscale("log", base=2)
ax.set_xticks(BUDGETS)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("token budget (max_new_tokens inside the think block)")
ax.set_ylabel("share of problems solved (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
ax.set_title(f"MATH-500 levels 4-5, n={CURVES[PRIMARY][0]['n']}, greedy, budget forced at the cap",
             fontsize=9, loc="left", color="#555")

fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2_curve.png", dpi=160)
plt.show()


In [ ]:
# Write down what the run produced before deciding what the caption says about it.
KNEE_GAIN_PTS = 2.0
BEND = {}
for mid in SWEPT:
    pts = CURVES[mid]
    gains = [dict(from_budget=pts[i - 1]["budget"], to_budget=pts[i]["budget"],
                  gain_pts=(pts[i]["accuracy"] - pts[i - 1]["accuracy"]) * 100)
             for i in range(1, len(pts))]
    monotone = all(g["gain_pts"] >= 0 for g in gains)
    # "The bend" = the first doubling after which no later doubling buys KNEE_GAIN_PTS.
    # Resolution-dependent: at n=160 one problem is 0.6 points, at n=20 it is 5, so in
    # QUICK this degenerates to "accuracy never rises again". Read the gains table.
    knee = None
    for i, g in enumerate(gains):
        if all(h["gain_pts"] < KNEE_GAIN_PTS for h in gains[i:]):
            knee = g["from_budget"]
            break
    BEND[mid] = dict(gains=gains, monotone=monotone, knee_budget=knee,
                     ceiling=max(p["accuracy"] for p in pts),
                     floor=pts[0]["accuracy"],
                     total_range_pts=(max(p["accuracy"] for p in pts) - pts[0]["accuracy"]) * 100)

    print(f"\n{mid}")
    for g in gains:
        bar = "#" * max(int(round(g["gain_pts"])), 0)
        print(f"  {g['from_budget']:>5} -> {g['to_budget']:<5} {g['gain_pts']:>+6.1f} pts  {bar}")
    print(f"  floor {pts[0]['accuracy']:.0%} at {pts[0]['budget']} tokens, "
          f"ceiling {BEND[mid]['ceiling']:.0%}, span {BEND[mid]['total_range_pts']:.0f} pts")
    print(f"  bends at: {knee if knee else 'has not bent inside this grid -- extend it'}")
    if not monotone:
        dips = [g for g in gains if g["gain_pts"] < 0]
        print(f"  NON-MONOTONE: {len(dips)} doubling(s) lose accuracy "
              f"({', '.join(str(d['to_budget']) for d in dips)}).")


## 14. Figure 3 — the flip-point strip

One row per problem, sorted by flip point, shaded from the budget where it first grades correct.

A problem has a **threshold**, not a gradient, so each row is a step and figure 2's smooth climb
is the average of a few hundred steps landing in different places. Per-item **regressions**,
correct at one budget and wrong at a larger one, are marked rather than hidden, and they are why
flip point is defined as *first* correct rather than *permanently* correct.

In [ ]:
FLIP_MODEL = PRIMARY
rows = by_problem(FLIP_MODEL)

flip_items = []
for pid, row in rows.items():
    verdicts = [bool(row[b]["correct"]) for b in BUDGETS]
    fc = next((BUDGETS[j] for j, v in enumerate(verdicts) if v), None)
    # "monotone" = once it flips, it stays flipped.
    stable = all(verdicts[BUDGETS.index(fc):]) if fc is not None else True
    flip_items.append(dict(
        problem_id=pid,
        level=row[MAX_BUDGET]["level"],
        subject=row[MAX_BUDGET]["subject"],
        correct_by_budget=verdicts,
        first_correct_budget=fc,
        monotone=stable,
        tokens_at_max=row[MAX_BUDGET]["tokens_generated"],
        truncated_at_max=row[MAX_BUDGET]["truncated"],
    ))

# never-flips sort to the bottom, then by flip point, then by how long they took
flip_items.sort(key=lambda d: (d["first_correct_budget"] is None,
                               BUDGETS.index(d["first_correct_budget"]) if d["first_correct_budget"] else 0,
                               d["tokens_at_max"]))

never = [d for d in flip_items if d["first_correct_budget"] is None]
regress = [d for d in flip_items if not d["monotone"]]
hist = {b: sum(d["first_correct_budget"] == b for d in flip_items) for b in BUDGETS}

print(f"{FLIP_MODEL}, n={len(flip_items)}\n")
print(f"  {'flips at':>10}{'problems':>10}   distribution")
for b in BUDGETS:
    print(f"  {b:>10}{hist[b]:>10}   {'#' * hist[b]}")
print(f"  {'never':>10}{len(never):>10}   {'#' * len(never)}")
print(f"\n{len(never)} of {len(flip_items)} ({len(never)/len(flip_items):.0%}) are never reached by any "
      f"budget in this grid.")
print(f"{len(regress)} problem(s) go correct and then wrong again at a larger budget.")


In [ ]:
import numpy as np
from matplotlib.colors import ListedColormap

grid = np.array([[1 if v else 0 for v in d["correct_by_budget"]] for d in flip_items])

fig, (ax, axh) = plt.subplots(1, 2, figsize=(10, 5), sharey=False,
                              gridspec_kw=dict(width_ratios=[3, 1], wspace=0.25))

ax.imshow(grid, aspect="auto", interpolation="nearest",
          cmap=ListedColormap(["#e8e6e1", "#3b4a6b"]), vmin=0, vmax=1)
ax.set_xticks(range(len(BUDGETS)))
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("token budget")
ax.set_ylabel(f"problems, sorted by flip point  (n={len(flip_items)})")
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# regressions: correct, then wrong again at a larger budget
for y, d in enumerate(flip_items):
    if not d["monotone"]:
        for x, v in enumerate(d["correct_by_budget"]):
            if not v and any(d["correct_by_budget"][:x]):
                ax.plot(x, y, marker="x", ms=4, color="#b23", mew=1.2)

if never:
    ax.axhline(len(flip_items) - len(never) - 0.5, color="#b23", lw=1.2, ls="--")
    ax.text(len(BUDGETS) - 0.4, len(flip_items) - len(never) / 2,
            f"never flips\n{len(never)} problems", color="#b23", fontsize=8, va="center")

axh.barh(range(len(BUDGETS)), [hist[b] for b in BUDGETS], color="#3b4a6b", height=0.6)
axh.barh([len(BUDGETS)], [len(never)], color="#b23", height=0.6)
axh.set_yticks(range(len(BUDGETS) + 1))
axh.set_yticklabels([str(b) for b in BUDGETS] + ["never"], fontsize=8)
axh.invert_yaxis()
axh.set_xlabel("problems")
axh.set_title("flip point", fontsize=9, loc="left", color="#555")
axh.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"{model_meta[FLIP_MODEL]['label']} — the budget at which each problem first grades correct",
             fontsize=10, x=0.02, ha="left", color="#333")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_3_flip_strip.png", dpi=160)
plt.show()


## 15. Figure 4 — what the extra thinking bought

Accuracy against the number of injections, on the same items as everything above.

**The claim this figure is allowed to make is bounded by what it measures:** the word appended to
a model that was not trained to deliberate, at the end of its turn rather than inside a thinking
block. Whatever it shows is a fact about these checkpoints and this intervention, and the cell
after the plot counts what moved in each direction rather than reporting a net.


In [ ]:
def wait_curve_for(mid):
    rs = WAIT_RESULTS[mid]
    pts = []
    for k in WAITS:
        at_k = [r for r in rs if r["waits"] == k]
        pts.append(dict(waits=k, n=len(at_k),
                        n_correct=sum(r["correct"] for r in at_k),
                        accuracy=mean([r["correct"] for r in at_k]),
                        mean_tokens=mean([r["own_tokens"] for r in at_k]),
                        total_tokens=sum(r["own_tokens"] for r in at_k),
                        offered_a_stop=mean([r["offered_a_stop"] for r in at_k]),
                        carried=mean([r["carried_from"] is not None for r in at_k])))
    return pts


WAIT_CURVES, WAIT_PAIRED = {}, {}
for mid in WAIT_RESULTS:
    WAIT_CURVES[mid] = wait_curve_for(mid)
    by_k = {}
    for r in WAIT_RESULTS[mid]:
        by_k.setdefault(r["waits"], {})[r["problem_id"]] = r
    base = by_k[WAITS[0]]
    WAIT_PAIRED[mid] = []
    for k in WAITS[1:]:
        pids = sorted(set(base) & set(by_k[k]))
        fixed = [p for p in pids if by_k[k][p]["correct"] and not base[p]["correct"]]
        broken = [p for p in pids if base[p]["correct"] and not by_k[k][p]["correct"]]
        # Exact McNemar over the items where the two disagree, as page 5 uses. A net
        # of zero with both columns large is a different result from nothing moving.
        n_fixed, n_broken, p_value = harness.mcnemar(
            {q: by_k[k][q]["correct"] for q in pids},
            {q: base[q]["correct"] for q in pids})
        assert (n_fixed, n_broken) == (len(fixed), len(broken))
        WAIT_PAIRED[mid].append(dict(waits=k, fixed=n_fixed, broken=n_broken,
                                     p=p_value, n=len(pids),
                                     fixed_ids=fixed, broken_ids=broken))

for mid in WAIT_CURVES:
    print(f"\n{mid}   n={WAIT_CURVES[mid][0]['n']} items, MATH-500 levels 4-5, greedy")
    print(f"  {'waits':>7}{'solved':>10}{'accuracy':>10}{'mean tok':>10}{'stops':>8}"
          f"{'fixed':>8}{'broken':>8}")
    paired = {d["waits"]: d for d in WAIT_PAIRED[mid]}
    for p in WAIT_CURVES[mid]:
        d = paired.get(p["waits"])
        print(f"  {p['waits']:>7}{p['n_correct']:>6}/{p['n']:<3}{p['accuracy']:>9.1%}"
              f"{p['mean_tokens']:>10,.0f}{p['offered_a_stop']:>8.0%}"
              + (f"{d['fixed']:>8}{d['broken']:>8}" if d else f"{'-':>8}{'-':>8}"))
    print("  fixed/broken are against k=0 on the same item, so they are paired counts,")
    print("  and a net of zero with both columns large is a different result from nothing.")


In [ ]:
import matplotlib.pyplot as plt

# A published trace file made before this arm existed replays fine and draws
# everything else; it simply has no fourth figure.
if not WAIT_CURVES:
    print("no wait-arm records in this run, so figure 4 is not drawn")
_LEAD = PRIMARY if PRIMARY in WAIT_CURVES else next(iter(WAIT_CURVES), None)

if WAIT_CURVES:
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for (mid, pts), colour in zip(WAIT_CURVES.items(), ["#3b4a6b", "#a2704a", "#5c7a5c"]):
        ax.plot([p["waits"] for p in pts], [p["accuracy"] * 100 for p in pts],
                marker="o", ms=4.5, lw=2, color=colour, label=model_meta[mid]["label"])
    ax.set_xticks(WAITS)
    ax.set_xlabel(f'number of "{WAIT_WORD}" injections')
    ax.set_ylabel("share of problems solved (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="lower right")
    ax.set_title(f"MATH-500 levels 4-5, n={WAIT_CURVES[_LEAD][0]['n']}, greedy, "
                 f'"{WAIT_WORD}" appended when it tries to stop',
                 fontsize=9, loc="left", color="#555")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_4_waits.png", dpi=160)
    plt.show()


In [ ]:
# Two traces the page draws: one the extra thinking rescued, one it wrecked. Both are
# curated from the run and flagged when the run does not contain one, because a page
# may not show a specimen that had to be invented.
def wait_rows(mid):
    out = {}
    for r in WAIT_RESULTS[mid]:
        out.setdefault(r["problem_id"], {})[r["waits"]] = r
    return out


WAIT_SPECIMENS = {"fixed": None, "broken": None, "is_fallback": {}}
WAIT_BY_PID = {}
if WAIT_RESULTS:
    _p = PRIMARY if PRIMARY in WAIT_RESULTS else sorted(WAIT_RESULTS)[0]
    WAIT_BY_PID = wait_rows(_p)

    def _pick(want_fixed):
        best = None
        for pid, by_k in WAIT_BY_PID.items():
            if set(WAITS) - set(by_k):
                continue
            v = [by_k[k]["correct"] for k in WAITS]
            hit = (not v[0] and any(v[1:])) if want_fixed else (v[0] and not all(v))
            if not hit:
                continue
            # prefer the shortest chain, so the page can show the whole thing
            size = by_k[WAITS[-1]]["own_tokens"]
            if best is None or size < best[0]:
                best = (size, pid)
        return best[1] if best else None

    for kind, want in (("fixed", True), ("broken", False)):
        pid = _pick(want)
        WAIT_SPECIMENS["is_fallback"][kind] = pid is None
        if pid is None:
            print(f"no problem was {'rescued' if want else 'wrecked'} by the injections; "
                  f"derived.json flags it and the page must not draw one")
        WAIT_SPECIMENS[kind] = pid

    for kind in ("fixed", "broken"):
        pid = WAIT_SPECIMENS[kind]
        if pid is None:
            continue
        by_k = WAIT_BY_PID[pid]
        print("\n" + "=" * 100)
        print(f"{kind}: {pid}   ground truth {by_k[WAITS[0]]['ground_truth']!r}")
        for k in WAITS:
            r = by_k[k]
            print(f"  waits {k}   {r['own_tokens']:>5} tok   answer {r['extracted_answer']!r:<24}"
                  f"{'CORRECT' if r['correct'] else 'wrong'}"
                  f"{'   (carried)' if r['carried_from'] is not None else ''}")


## 16. The handoff to page 7

Same total budget, spent differently: instead of one chain of `MAX_BUDGET` tokens, N chains of
`BUDGETS[0]`. The numbers below are what page 7's run is measured against.

The cell also overlays **page 5's two endpoints** on this curve, if that run's artifact is
present. Page 5's forced-blurt condition is this curve at a budget of about six tokens, and its
bare condition is the same curve unconstrained, on the same model, items and grader. They are not
a separate result to reconcile; they are the ends of the segment measured here.

In [ ]:
top = next(p for p in CURVES[PRIMARY] if p["budget"] == MAX_BUDGET)
bot = CURVES[PRIMARY][0]
n_chains = MAX_BUDGET // BUDGETS[0]

HANDOFF = {
    "model_id": PRIMARY,
    "max_budget": MAX_BUDGET,
    "min_budget": BUDGETS[0],
    "chains_at_min_budget": n_chains,
    "accuracy_at_max": top["accuracy"],
    "accuracy_at_min": bot["accuracy"],
    "total_tokens_at_max": top["total_tokens"],
    "mean_tokens_at_max": top["mean_tokens"],
    "never_flip_share": len(never) / len(flip_items),
    "never_flip_count": len(never),
    "n_items": len(flip_items),
}

print(f"one chain of {MAX_BUDGET:,} tokens     -> {top['accuracy']:.0%} solved "
      f"({top['mean_tokens']:,.0f} tokens/problem actually spent)")
print(f"one chain of {BUDGETS[0]:,} tokens      -> {bot['accuracy']:.0%} solved")
print(f"\nthe same budget buys {n_chains} chains of {BUDGETS[0]} tokens. Page 7 asks what {n_chains} "
      f"voters at {bot['accuracy']:.0%} each are worth.")
print(f"\nand {len(never)} of {len(flip_items)} problems ({len(never)/len(flip_items):.0%}) are not "
      f"reached by any budget here, so length is not the only axis.")


# --- page 5's endpoints, if that run is on disk -----------------------------
P5 = Path("out/page_05/derived.json")
ENDPOINTS = None
if P5.exists():
    p5 = json.loads(P5.read_text())
    by_cond = {c["condition"]: c for c in p5["figure_2"]["conditions"]}
    if {"forced_blurt", "bare"} <= set(by_cond):
        ENDPOINTS = {"source": "page_05/derived.json",
                     "run_id": p5["manifest"].get("run_id"),
                     "model": p5["manifest"]["models"][0]["label"]
                              if p5["manifest"].get("models") else None,
                     "suppressed": by_cond["forced_blurt"]["accuracy"],
                     "suppressed_tokens": by_cond["forced_blurt"].get("median_tokens"),
                     "unconstrained": by_cond["bare"]["accuracy"],
                     "unconstrained_tokens": by_cond["bare"].get("median_tokens")}
        print(f"\npage 5, same model and items:")
        print(f"  budget ~{ENDPOINTS['suppressed_tokens']} tokens (forbidden to reason)  "
              f"-> {ENDPOINTS['suppressed']:.0%}")
        print(f"  unconstrained ({ENDPOINTS['unconstrained_tokens']} tokens median)      "
              f"-> {ENDPOINTS['unconstrained']:.0%}")
        print(f"  this page fills the budgets between them: {BUDGETS}")
    else:
        print("\npage 5's derived.json has no forced_blurt/bare pair; no overlay drawn")
else:
    print("\nout/page_05/derived.json not found; run page 5 to overlay its endpoints")
HANDOFF["page_5_endpoints"] = ENDPOINTS


## 17. Artifacts

`derived.json` is the contract with the page: the page reads it and nothing else, and **no
inference runs at demonstration time.**

**Which traces get bundled.** Figure 1's specimen at every budget, plus `N_BUNDLED_ROWS` more
problems spanning the flip-point range, because figure 3's rows are clickable and a click loads
that problem into figure 1. Everything else stays in `traces.jsonl`, which ships as a download.

In [ ]:
N_BUNDLED_ROWS = 8          # figure 3's clickable rows, spanning the flip-point range


def bundle_row(pid):
    row = PRIMARY_BY_PROBLEM[pid]
    ref = row[MAX_BUDGET]
    return {
        "problem_id": pid,
        "level": ref["level"],
        "subject": ref["subject"],
        "question": ref["question"],
        "ground_truth": ref["ground_truth"],
        "first_correct_budget": first_correct(row),
        # what the model would have written past the cut -- figure 1's greyed text
        "reference_tokens": ref["tokens"][: ref["cut_index"]] if ref["forced"] else ref["tokens"],
        "reference_truncated": ref["truncated"],
        "by_budget": [
            {k: row[b][k] for k in ("budget", "trace", "tokens", "cut_index", "answer_start",
                                    "truncated", "forced", "forced_suffix", "answer_closed",
                                    "extracted_answer", "correct", "own_tokens",
                                    "answer_tokens", "tokens_generated")}
            for b in BUDGETS
        ],
    }


# span the sorted strip so the bundled rows include early, late and never-flipping problems
idx = sorted({round(i * (len(flip_items) - 1) / max(N_BUNDLED_ROWS - 1, 1))
              for i in range(N_BUNDLED_ROWS)})
bundled_ids = [flip_items[i]["problem_id"] for i in idx]
if SPECIMEN_ID not in bundled_ids:
    bundled_ids.insert(0, SPECIMEN_ID)

WAIT_BUNDLED = [p for p in (WAIT_SPECIMENS["fixed"], WAIT_SPECIMENS["broken"]) if p]


def wait_bundle_row(pid):
    by_k = WAIT_BY_PID[pid]
    ref = by_k[WAITS[-1]]
    return {
        "problem_id": pid,
        "level": ref["level"],
        "subject": ref["subject"],
        "question": ref["question"],
        "ground_truth": ref["ground_truth"],
        "first_correct_waits": next((k for k in WAITS if by_k[k]["correct"]), None),
        "by_waits": [
            {k2: by_k[k][k2] for k2 in ("waits", "trace", "tokens", "wait_spans", "cut_index",
                                        "answer_start", "truncated", "forced", "forced_suffix",
                                        "answer_closed", "offered_a_stop", "extracted_answer",
                                        "correct", "own_tokens", "answer_tokens",
                                        "tokens_generated", "carried_from")}
            for k in WAITS
        ],
    }


print(f"bundling {len(bundled_ids)} problems x {len(BUDGETS)} budgets:")
for pid in bundled_ids:
    fc = first_correct(PRIMARY_BY_PROBLEM[pid])
    print(f"  {pid:<34} flips at {str(fc) if fc else 'never':>6}"
          f"{'   <- figure 1 specimen' if pid == SPECIMEN_ID else ''}")
print(f"\nbundling {len(WAIT_BUNDLED)} problems x {len(WAITS)} wait steps:")
for pid in WAIT_BUNDLED:
    kind = "rescued" if pid == WAIT_SPECIMENS["fixed"] else "wrecked"
    print(f"  {pid:<34} {kind} by the injections")


In [ ]:
git_sha = harness.git_sha
repo_revision = harness.repo_revision
notebook_sha256 = lambda: harness.file_sha256("page_06_thinking_longer.ipynb")


manifest = {
    "run_id": RUN_ID,
    "notebook": "page_06_thinking_longer.ipynb",
    "page": 6,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": git_sha(),
    "notebook_sha256": notebook_sha256(),
    "models": [
        {"id": mid,
         "label": model_meta[mid]["label"],
         "params": model_meta[mid]["params"],
         "revision": repo_revision(mid),
         "dtype": model_meta[mid]["dtype"],              # as loaded, not as requested
         "tokenizer_sha256": model_meta[mid]["tokenizer_sha256"],
         "role": "primary" if mid == PRIMARY else "second curve",
         "quant": model_meta[mid]["quant"]}
        for mid in SWEPT
    ],
    "dtype_requested": str(DTYPE) if DTYPE is not None else None,
    "replayed": RUN_MODE == "REPLAY",
    "decoding": {"strategy": "greedy", "do_sample": False,
                 "temperature": None, "top_p": None, "top_k": None},
    # greedy argmax is only deterministic up to floating-point reduction order, and
    # ch02's get_device() enables TF32 on CUDA, so this belongs in the record.
    "tf32_matmul": bool(getattr(getattr(torch.backends, "cuda", None), "matmul", None)
                        and torch.backends.cuda.matmul.allow_tf32),
    "budgets": BUDGETS,
    "waits": {"word": WAIT_WORD, "grid": WAITS, "allowance": WAIT_ALLOWANCE,
              "injection": WAIT_TEXT,
              "where": "appended when the model tries to end its turn; these checkpoints "
                       "have no thinking block, so that is after a finished answer"},
    "forcing": {"stem": FORCE_STEM, "answer_allowance": ANSWER_ALLOWANCE},
    "items": {
        "source": "MATH-500 (ch03 copy)" if DATASET.startswith("math") else "GSM8K test split",
        "dataset": DATASET,
        "n": len(items),
        "item_set_sha256": ITEM_SET_HASH,
        "seed": SEED,
    },
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "prompt_template": render_prompt("{question}"),
    "chat_template": chat if RUN_MODE != "REPLAY" else None,
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device) if device is not None else None},
    "integrity": INTEGRITY,
}

derived = {
    "manifest": manifest,
    "budgets": BUDGETS,
    # The one line printed under figure 2.
    "procedure": (
        f"Cap generation at N tokens; if the model is still inside its think block, "
        f"close it and append \"Final answer: \\boxed{{\"; allow {ANSWER_ALLOWANCE} more "
        f"tokens; grade with MATH-500's reference answer. Repeat for "
        f"N in {{{', '.join(str(b) for b in BUDGETS)}}} over {len(items)} problems "
        f"from levels 4-5, greedy throughout."),
    "pilot": PILOT,
    "figure_1": {
        "specimen_id": SPECIMEN_ID,
        "specimen_is_fallback": SPECIMEN_IS_FALLBACK,
        "flip_budget": SPECIMEN_FLIP,
        "n_candidate_specimens": len(candidates),
        "rows": [bundle_row(pid) for pid in bundled_ids],
        "inset": INSET,
    },
    "figure_2": {
        "curves": [{"model_id": mid,
                    "label": model_meta[mid]["label"],
                    "params": model_meta[mid]["params"],
                    "points": CURVES[mid],
                    **BEND[mid]}
                   for mid in SWEPT],
    },
    "figure_3": {
        "model_id": FLIP_MODEL,
        "items": flip_items,
        "histogram": {str(b): hist[b] for b in BUDGETS},
        "never_flip_count": len(never),
        "never_flip_share": len(never) / len(flip_items),
        "n_regressions": len(regress),
        # the rows whose full traces are bundled: clicking one loads it into figure 1
        "clickable_row_ids": bundled_ids,
    },
    "figure_wait": None if not WAIT_CURVES else {
        "waits": WAITS,
        "word": WAIT_WORD,
        "curves": [{"model_id": mid,
                    "label": model_meta[mid]["label"],
                    "params": model_meta[mid]["params"],
                    "points": WAIT_CURVES[mid],
                    "paired": WAIT_PAIRED[mid]}
                   for mid in WAIT_CURVES],
        "specimen_fixed": WAIT_SPECIMENS["fixed"],
        "specimen_broken": WAIT_SPECIMENS["broken"],
        "specimen_is_fallback": WAIT_SPECIMENS["is_fallback"],
        "rows": [wait_bundle_row(pid) for pid in WAIT_BUNDLED],
        "integrity": WAIT_CHECKS,
    },
    "handoff": HANDOFF,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

kb = lambda p: p.stat().st_size / 1024
print(f"derived.json   {kb(DERIVED_PATH):>10,.1f} KB   <- bundled with the page")
print(f"manifest.json  {kb(MANIFEST_PATH):>10,.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {kb(TRACES_PATH):>10,.1f} KB   <- published as a download "
      f"({len(carried)} of {len(all_records)} records are carry-forward duplicates)")

if kb(DERIVED_PATH) > 1024:
    print("\n^ derived.json is over 1 MB. Lower N_BUNDLED_ROWS: the page bundles what it draws "
          "and links the rest.")
if kb(TRACES_PATH) > 50 * 1024:
    print("\n^ traces.jsonl is over 50 MB, which is a download rather than an asset. Page 7's "
          "N-sweep\n  will be larger again; if this is uncomfortable now, that is the signal to "
          "compress or to\n  store carry-forward records as pointers before page 7 forces it.")

print(f"\nNot committed by this notebook: out/ is gitignored. Promoting a run to the page is "
      f"pipeline\nstep 4 -- copy derived.json and manifest.json into web/data/, where the "
      f"cross-language\nagreement test recomputes these same aggregates in TypeScript and asserts "
      f"they match.")


## 18. Notes for re-derivation

- Greedy decoding diverges on a handful of items across hardware (`get_device` enables TF32 on
  CUDA): expect the shape to reproduce, not every digit.
- Two gates can only be answered after a `FULL` run, and both were printed above: whether the
  curve bends inside the grid (`figure_2.knee_budget`), and whether the never-flipping band is
  big enough to see and small enough to be a band (`figure_3.never_flip_share`).
- Changing the item set, dtype or device changes `RUN_ID`; old traces stay as a record and are
  not replayed.

What the page may claim is [`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §7.